<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Logistic Regression

*Session 5 · Notebook 03.07 · Lecture · Student version*

## Overview

Logistic regression is the classic **credit scorecard** model: it predicts the probability of a binary outcome (here, whether a loan is not fully repaid) and, crucially, its coefficients can be read as **odds ratios**, so it is both accurate and explainable. This notebook builds a logistic regression end to end, evaluates it the way a risk team would (confusion matrix, precision/recall, ROC and AUC), **interprets** the fitted model, and shows how to move the **decision threshold** when the classes are imbalanced.

We use the LendingClub `loan_data` dataset (9,578 loans), where about 16% were not fully paid.

## Learning Objectives

By the end of this notebook you will be able to:

- Explain how logistic regression models a probability with the logistic (sigmoid) function.
- Train a logistic regression inside a preprocessing pipeline.
- Evaluate it with a confusion matrix, precision/recall, and the ROC curve / AUC.
- Interpret the coefficients as odds ratios (the scorecard's key selling point).
- Adjust the decision threshold to trade off precision and recall on an imbalanced target.

## Prerequisites

- Session 5 notebooks 01.02, 01.03 and 03.02 (the scikit-learn workflow, preprocessing pipelines, classification evaluation).
- Session 4 (probability, odds).

<a id="setup"></a>
# Section 0: Setup

We use `loan_data.csv` (LendingClub loans), read from the repo-root `datasets/` folder (two levels up). The target is `not.fully.paid` (1 = the loan was not fully repaid). We rename the dotted column names to underscores so they are easier to work with.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, RocCurveDisplay, roc_auc_score)

sns.set_theme(style='whitegrid')
np.random.seed(42)

# Or read directly from the public S3 bucket (no local file needed):
# loans = pd.read_csv('https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/Data_Sources_CBS_Risk/Session_5/loan_data.csv')
# ...or read the paths from a config file (the local read below stays the default):
# from config import session_datasets_http
# loans = pd.read_csv(session_datasets_http["loan_data"])
# Or from S3 with Spark, then to pandas (needs a SparkSession, e.g. on Databricks):
# loans = spark.read.csv("s3://rockborne-bucket-01-cbs/Data_Sources_CBS_Risk/Session_5/loan_data.csv", header=True, inferSchema=True).toPandas()
loans = pd.read_csv('../../datasets/Session_5/loan_data.csv')
loans.columns = loans.columns.str.replace('.', '_', regex=False)
print('shape:', loans.shape)
print('target (not_fully_paid):', loans['not_fully_paid'].value_counts().to_dict())
loans.head()

<a id="sec1"></a>
# Section 1: Why this matters for risk analysis

Logistic regression is, for many risk teams, *the* model. It is the workhorse behind credit scorecards for good reasons.

| Property | Why a risk team values it |
|---|---|
| Outputs a probability | A probability of default is exactly what pricing, limits and cut-offs need |
| Coefficients are odds ratios | Every driver has a clear, explainable effect ("higher FICO lowers the odds of default by X%") |
| Transparent and auditable | Regulators can see and challenge each coefficient, unlike a black-box model |
| Adjustable threshold | The cut-off can be tuned to the business appetite for risk |
| Fast and stable | Trains quickly and behaves predictably, ideal for regular re-fitting |

It is the natural baseline (and often the deployed model) against which trees, forests and SVMs are compared.

<a id="sec2"></a>
# Section 2: What is logistic regression?

**Definition:** logistic regression models the **probability** that a binary outcome is 1. It fits a linear combination of the features and passes it through the **logistic (sigmoid)** function, which squashes any number into the range 0 to 1.

**Example:** the probability that a loan is not repaid, given the borrower's FICO score, debt-to-income ratio, interest rate and so on.

**Analogy:** linear regression draws a straight line, which for a 0/1 outcome would run below 0 and above 1 (impossible probabilities). Logistic regression bends that line into an S-shaped dimmer switch that always stays between 0 and 1.

**Explanation:**

- It is linear in the **log-odds**: `log(p / (1 - p)) = b0 + b1*x1 + b2*x2 + ...`.
- The sigmoid then converts the log-odds back into a probability `p`.
- Each coefficient `b` is the change in log-odds per unit of the feature; **`exp(b)` is the odds ratio**, the multiplicative effect on the odds. An odds ratio above 1 raises the odds of the outcome; below 1 lowers them.
- A **decision threshold** (default 0.5) turns the probability into a class label, and it can be moved to suit the business.

**scikit-learn documentation:** [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)

### The sigmoid function

In [ ]:
# Your turn. Write your solution here:

<a id="sec3"></a>
# Section 3: Load and prepare the data

`purpose` is categorical (the reason for the loan); everything else is numeric. There are no missing values. We standardise the numeric features (so coefficients are comparable and the solver converges well) and one-hot encode `purpose`, all inside a `ColumnTransformer` so the preprocessing is fitted on the training data only.

In [ ]:
# Your turn. Write your solution here:

<a id="sec4"></a>
# Section 4: Train a logistic regression

We build the preprocessing and the model into one pipeline and fit it. `max_iter=1000` gives the solver room to converge.

In [ ]:
# Your turn. Write your solution here:

### The model outputs a probability

A logistic regression outputs a **probability**, not a label: `predict_proba` returns, for each loan, the probability it is repaid (column 0) and the probability it is not fully paid (column 1). The default 0.5 threshold then turns that probability into a class. Here are the model's default probabilities for the first few test loans, next to what actually happened.

In [ ]:
# Your turn. Write your solution here:

<a id="sec5"></a>
# Section 5: Evaluate the model

Accuracy looks high, but the classes are imbalanced (about 84% of loans are repaid), so a model that predicts 'repaid' for almost everyone can score well on accuracy while catching very few of the loans that actually default. The confusion matrix and the recall for class 1 reveal this.

### How to read a classification report

The `classification_report` below has one row per class. The columns mean:

- **precision:** of the cases the model *labelled* as this class, what fraction truly were. "When it predicts default, how often is it right?"
- **recall:** of the cases that *truly are* this class, what fraction the model caught. "Of the real defaults, how many did it find?"
- **f1-score:** the harmonic mean of precision and recall, balancing the two in one number (1.0 perfect, 0 useless).
- **support:** how many real cases of that class exist (the row count).

And at the bottom:

- **accuracy:** the overall fraction of predictions that were correct.
- **macro avg:** the plain average across classes (each class counts equally, so a rare class cannot hide).
- **weighted avg:** the average weighted by support (dominated by the larger class).

In [ ]:
# Your turn. Write your solution here:

### Reading this report

On this imbalanced data (about 84% of loans are repaid) the danger of judging by accuracy jumps out:

- **`repaid` (support 1609):** precision 0.84, recall 0.99. The model labels almost every loan 'repaid' and is right most of the time, simply because about 84% of loans really are repaid.
- **`not fully paid` (support 307):** recall is just **0.05**. Of the 307 loans that actually defaulted, the model flags only about 5% (roughly 15 of them). Its precision of 0.52 says that on the rare occasion it does flag a default it is right about half the time, but it almost never flags one.
- **accuracy 0.84** looks healthy, but it is essentially just the proportion of repaid loans: a model that blindly predicted 'repaid' for everyone would score about the same. This is exactly why accuracy alone is misleading on imbalanced data.
- **macro avg recall 0.52** (the unweighted average across the two classes) is barely above a coin flip, and it exposes what accuracy hides: the model is failing at the one job that matters, catching defaulters.

The cause is the default **0.5 threshold** combined with the class imbalance. Section 8 fixes it by lowering the threshold.

<a id="sec6"></a>
# Section 6: ROC curve and AUC

Because a logistic regression outputs probabilities, the right way to judge it independently of any single threshold is the **ROC curve** and its area, **AUC**. The ROC plots the true positive rate against the false positive rate across every threshold; AUC summarises it in one number (0.5 = random, 1.0 = perfect). AUC is the standard headline metric for scorecards.

In [ ]:
# Your turn. Write your solution here:

<a id="sec7"></a>
# Section 7: Interpreting the model - odds ratios

This is logistic regression's superpower: unlike most classifiers, its coefficients translate into plain-language statements about risk. To read them, we need three ideas.

**Probability vs odds.** These are two ways to express the same chance:

- **Probability** `p`: the chance of default, from 0 to 1.
- **Odds** `p / (1 - p)`: the chance of default relative to not defaulting. p = 0.5 is odds of 1 (evens, 1 to 1); p = 0.2 is odds of 0.25 (1 to 4 against); p = 0.8 is odds of 4 (4 to 1 on).

**The coefficient.** Logistic regression is linear in the **log-odds**: `log(p / (1 - p)) = b0 + b1*x1 + b2*x2 + ...`. So each coefficient `b` is the change in the log-odds of default for a one-unit increase in that feature, holding the others constant. Log-odds are hard to picture, so we exponentiate them.

**The odds ratio.** `exp(b)` is the **odds ratio**: the number you *multiply* the odds of default by for a one-unit increase in the feature. Above 1 raises the odds, below 1 lowers them, exactly 1 means no effect. Because we standardised the numeric features, "one unit" here is **one standard deviation**; for `purpose`, the odds ratio is measured relative to the dropped reference category.

We pull the coefficients out of the fitted pipeline, exponentiate them into odds ratios, and sort.

In [ ]:
# Your turn. Write your solution here:

### Reading the table

- An **odds ratio above 1** means the feature *raises* the odds that a loan is not fully paid; **below 1** means it *lowers* them.
- For example, a higher `fico` score has an odds ratio well below 1 (safer borrowers, lower default odds), while a higher `int_rate` has an odds ratio above 1 (loans priced as riskier do default more). Loan `purpose` of `small_business` typically carries higher default odds than the reference purpose.
- This is exactly the kind of explanation a credit committee or regulator can read and challenge, which is why logistic regression remains the scorecard standard.

**Worked examples from the table above** (remember these multiply the *odds*, and the numeric features are per one standard deviation because we standardised them):

- `purpose_small_business` has an odds ratio of about **1.74**: a small-business loan has roughly 74% higher odds of default than the reference purpose, all else equal. It is the single biggest riser.
- `fico` has an odds ratio of about **0.73**: each one-standard-deviation increase in the FICO score multiplies the odds of default by 0.73, cutting them by roughly a quarter. Safer borrowers, as expected.
- `purpose_credit_card` has an odds ratio of about **0.60**: these loans carry the lowest default odds relative to the reference purpose.
- `int_rate` has an odds ratio of about **1.05**: once the other features are accounted for, a one-standard-deviation rise in the interest rate lifts the odds only slightly.

**One more time, because it is the usual stumbling block:** an odds ratio multiplies the **odds**, not the probability. An odds ratio of 1.74 does not mean "74% more likely" in probability terms; it means the odds (`p / (1 - p)`) are multiplied by 1.74. Near the middle of the S-curve that is a large probability change; near the ends it is small.

### The odds ratio really is a multiplier

To make this concrete on the real model: take one test loan, then raise its `fico` score by exactly one standard deviation and re-predict. The odds of default should be multiplied by the `fico` odds ratio from the table above (about 0.73), no matter which loan we start from.

In [ ]:
# Your turn. Write your solution here:

<a id="sec8"></a>
# Section 8: Probabilities and the decision threshold

The model gives a probability; the **threshold** turns it into a decision. The default 0.5 is rarely the right business choice on imbalanced data. Lowering the threshold flags more loans as risky, which **catches more true defaulters (higher recall)** at the cost of more false alarms (lower precision). The right cut-off depends on the cost of a missed default versus the cost of a rejected good loan.

In [ ]:
# Your turn. Write your solution here:

<a id="sec9"></a>
# Section 9: A note on regularisation

Like Ridge and Lasso in the regression notebooks, logistic regression is **regularised**: it adds a penalty on the size of the coefficients to curb overfitting. scikit-learn's `LogisticRegression` applies **L2** regularisation by default.

**L1 vs L2** (the same two penalties you met in Lasso and Ridge):

| Aspect | L1 (like Lasso) | L2 (like Ridge, the default) |
|---|---|---|
| Penalty | sum of the absolute coefficients | sum of the squared coefficients |
| Effect on coefficients | can set some to exactly zero (feature selection) | shrinks all of them, none exactly zero |
| Use when | you want a small, explainable set of drivers | you want stable coefficients with correlated features |

Set it with `penalty='l1'`, `'l2'`, `'elasticnet'`, or `None`. L1 and elasticnet need a compatible solver such as `'liblinear'` or `'saga'`.

**`C` is the strength knob, and it is the inverse of `alpha`.** In Ridge and Lasso, a larger `alpha` meant *more* regularisation. In `LogisticRegression` (and SVM) the strength is `C = 1 / alpha`, so it runs the other way:

| Behaviour | `alpha` (Ridge / Lasso) | `C` (LogReg / SVM) |
|---|---|---|
| Bigger value means | more shrinkage | less shrinkage |
| Turn regularisation off | `alpha = 0` | `C` very large (e.g. 1e6) |
| Turn it up | large `alpha` | small `C` (e.g. 0.01) |

That is why the earlier demo used `C=1e6` to switch regularisation essentially off. The cells below show `C`'s effect first on performance, and then (exactly as the Ridge and Lasso notebooks did) on the coefficients themselves.

In [ ]:
# Your turn. Write your solution here:

### How C changes the coefficients (a coefficient path)

Just like the coefficient-path plots in the Ridge and Lasso notebooks, we can watch the L2 coefficients shrink as we strengthen the penalty. The only twist is the axis direction: because `C` is the *inverse* of `alpha`, strong regularisation is now on the **left** (small `C`) and weak regularisation on the **right** (large `C`).

In [ ]:
# Your turn. Write your solution here:

### L1 goes further: it zeroes features out

L2 only shrinks. L1 (like Lasso) can set coefficients to **exactly zero**, dropping features entirely. Here we compare, at a few values of `C`, how many of the 18 features each penalty keeps.

In [ ]:
# Your turn. Write your solution here:

<a id="exercises"></a>
# Section 10: Exercises

### Exercise 1: AUC

Compute the cross-validated AUC of the `logit` pipeline using 5-fold cross-validation (`cross_val_score(..., scoring='roc_auc', cv=5)`) on the full `X`, `y`. Report the mean.

In [ ]:
# Your turn. Write your solution here:


### Exercise 2: Interpret an odds ratio

From the `interpretation` table, find the odds ratio for `num__fico`. State in plain English what a one-standard-deviation increase in FICO does to the odds of default.

In [ ]:
# Your turn. Write your solution here:


### Exercise 3: Choose a threshold

Using the predicted probabilities `proba`, apply a threshold of 0.25 and print the recall on the defaulting class and how many loans get flagged.

In [ ]:
# Your turn. Write your solution here:


<a id="challenge"></a>
## Challenge (optional): logistic regression vs a random forest

Logistic regression is the interpretable baseline. Compare it to a random forest on this data using the metric that matters for a scorecard, **AUC**. Fit a `RandomForestClassifier` in a pipeline (it needs the same one-hot encoding but no scaling), compute both models' 5-fold cross-validated AUC, and comment on the accuracy-vs-interpretability trade-off.

In [ ]:
# Your turn. Write your solution here:


<a id="takeaways"></a>
## Key Takeaways

| Concept / command | What it does |
|---|---|
| Logistic regression | Models the probability of a binary outcome via the sigmoid |
| log-odds linear in features | `log(p/(1-p)) = b0 + b1x1 + ...` |
| `exp(coefficient)` = odds ratio | The multiplicative effect on the odds (>1 raises, <1 lowers) |
| `LogisticRegression(max_iter=1000)` | Fit the model (inside a preprocessing pipeline) |
| `predict_proba` | Probabilities, not just labels; the basis for a score |
| ROC curve / `roc_auc_score` | Threshold-independent evaluation; the scorecard headline metric |
| Decision threshold | Move it to trade recall (catch defaulters) against precision |
| `C`, L1/L2 penalty | Regularisation strength and type (same idea as Ridge/Lasso) |


## Conclusion

You can now build, evaluate and, above all, **interpret** a logistic regression: the sigmoid model, ROC/AUC evaluation, odds-ratio interpretation, and threshold tuning for imbalanced targets. This is the classic credit scorecard, and the interpretable baseline for every other classifier in this session.

<a id="reading"></a>
## Further Reading & Resources

- [scikit-learn: Logistic Regression](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression) the model and its options.
- [scikit-learn: ROC and AUC](https://scikit-learn.org/stable/modules/model_evaluation.html#roc-metrics) threshold-independent evaluation.
- [Odds ratios explained](https://en.wikipedia.org/wiki/Odds_ratio) interpreting logistic regression coefficients.